# 04 Baselines And Component Checks

Reviewer-facing baseline/component evidence notebook for the revision package.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Scope And Contracts

This notebook is intentionally CPU-friendly and artifact-first. It validates the frozen OOF and calibration outputs, writes reusable evidence tables, and records which fair baselines are complete or still blocked. It does not train models or run TBD heavy baseline commands.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import calibration_summary, git_commit, multilabel_metrics, save_csv, save_json, sha256_file

THRESHOLD = 0.5
N_BINS = 15
RUN_HEAVY_BASELINES = False

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
pred_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
run_manifest_path = revision_root / 'manifests' / f'{OOF_STEM}_prediction_run_manifest.json'
class_table_path = revision_root / 'tables' / f'{OOF_STEM}_class_summary.csv'

required_inputs = {
    'frozen_oof_predictions': pred_path,
    'oof_final_ema_freeze_manifest': freeze_path,
    'calibration_ci': calibration_path,
    'oof_run_manifest': run_manifest_path,
    'oof_class_summary': class_table_path,
}
for name, path in required_inputs.items():
    print(f'{name:24s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 02/03 artifacts are required before notebook 04: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before baseline/component checks.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 04 requires checkpoint_kind=final_ema in the freeze manifest.')

relative_pred = pred_path.as_posix()
frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
if relative_pred not in frozen_artifacts:
    raise ValueError(f'Freeze manifest does not include {relative_pred}')
pred_sha = sha256_file(pred_path)
if pred_sha != frozen_artifacts[relative_pred]['sha256']:
    raise RuntimeError('Prediction SHA256 changed after OOF freeze.')
if calibration.get('predictions_sha256') != pred_sha:
    raise RuntimeError('Calibration JSON does not match frozen OOF prediction SHA256.')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'threshold': THRESHOLD,
    'n_bins': N_BINS,
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'calibration_dataset': calibration.get('dataset'),
    'calibration_protocol': calibration.get('protocol'),
    'calibration_shape': calibration.get('shape'),
}
save_json(revision_root / 'manifests' / 'baseline_component_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'baseline_component_input_contract.json')


## Revision Plan Alignment

The cells below bind notebook 04 to the tracked reviewer plan. They make incomplete fair baselines explicit instead of silently treating placeholders as evidence.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_BASE', 'EXP_CAL', 'EXP_POOL'])]
relevant_claims = claims[claims['claim_id'].isin(['C01', 'C02', 'C05', 'C06'])]
relevant_tasks = tasks[tasks['id'].isin(['A2', 'A3', 'A4', 'B1'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'baseline_component_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'baseline_component_plan_alignment.json')


## No-Skill Reference Baselines

These are sanity references computed from the frozen OOF labels only. They are not substitutes for the fair model baselines requested by reviewers, but they help interpret whether the frozen model beats trivial prevalence/constant references under the same metric functions.


In [ ]:
with np.load(pred_path, allow_pickle=True) as data:
    y_true = np.asarray(data['y_true'], dtype=np.float32)
    y_prob_full = np.asarray(data['y_prob'], dtype=np.float32)
    class_names = data['class_names'].astype(str).tolist() if 'class_names' in data.files else list(range(y_true.shape[1]))

prevalence = y_true.mean(axis=0).astype(np.float32)
reference_predictions = {
    'full_ecg_ramba_frozen_oof': y_prob_full,
    'prevalence_probability_reference': np.tile(prevalence, (len(y_true), 1)).astype(np.float32),
    'zero_probability_reference': np.zeros_like(y_true, dtype=np.float32),
    'constant_0_5_probability_reference': np.full_like(y_true, 0.5, dtype=np.float32),
}

rows = []
for name, y_prob in reference_predictions.items():
    metrics = multilabel_metrics(y_true, y_prob, threshold=THRESHOLD)
    cal = calibration_summary(y_true, y_prob, n_bins=N_BINS)
    rows.append({
        'model_or_reference': name,
        'dataset': 'chapman_oof',
        'n_records': int(y_true.shape[0]),
        'n_classes': int(y_true.shape[1]),
        'threshold': THRESHOLD,
        'evidence_role': 'frozen_model' if name == 'full_ecg_ramba_frozen_oof' else 'no_skill_reference_not_fair_baseline',
        **metrics,
        **cal,
    })

reference_df = pd.DataFrame(rows)
reference_csv = revision_root / 'metrics' / 'no_skill_reference_baselines.csv'
reference_table = revision_root / 'tables' / 'table_no_skill_reference_baselines.csv'
reference_df.to_csv(reference_csv, index=False)
reference_df.to_csv(reference_table, index=False)
print('Wrote:', reference_csv)
print('Wrote:', reference_table)
display(reference_df)


## Fair Baseline Completion Matrix

This table is the reviewer-facing baseline ledger. A completed row must have the same split, preprocessing, threshold, and record-level protocol. Rows marked blocked must not be used to claim superiority over fair baselines.


In [ ]:
metric_names = [
    'roc_auc_macro', 'pr_auc_macro', 'f1_macro', 'f1_micro', 'precision_macro',
    'recall_macro', 'sensitivity_macro', 'specificity_macro', 'ppv_macro', 'npv_macro',
    'brier_macro', 'ece_macro', 'ece_max', 'mce_macro', 'mce_max'
]
full_metrics = {name: calibration.get('metrics', {}).get(name) for name in metric_names}
full_metrics.update({name: calibration.get('calibration', {}).get(name) for name in metric_names if name not in full_metrics or full_metrics[name] is None})

def baseline_row(name, family, status, evidence_path='', blocker='', notes=''):
    row = {
        'baseline_name': name,
        'model_family': family,
        'dataset': 'chapman_oof',
        'split_protocol': 'same frozen five-fold OOF required',
        'threshold': THRESHOLD,
        'aggregation_protocol': 'record-level fixed threshold; Q=3 for full model where applicable',
        'status': status,
        'evidence_path': evidence_path,
        'blocker': blocker,
        'notes': notes,
    }
    for metric in metric_names:
        row[metric] = full_metrics.get(metric) if name == 'Full ECG-RAMBA frozen OOF' else math.nan
    return row

baseline_rows = [
    baseline_row(
        'Full ECG-RAMBA frozen OOF',
        'full_model',
        'complete_frozen_oof',
        evidence_path=str(calibration_path),
        notes='Validated against oof_final_ema_freeze_manifest.json and calibration_ci_oof_final_ema_predictions.json.',
    ),
    baseline_row(
        'Raw Mamba', 'component_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    ),
    baseline_row(
        'ResNet1D/CNN', 'architecture_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    ),
    baseline_row(
        'MiniRocket-only', 'feature_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    ),
    baseline_row(
        'HRV-only', 'feature_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    ),
]

baseline_df = pd.DataFrame(baseline_rows)
baseline_csv = revision_root / 'metrics' / 'baseline_summary.csv'
baseline_table = revision_root / 'tables' / 'table_baselines.csv'
baseline_df.to_csv(baseline_csv, index=False)
baseline_df.to_csv(baseline_table, index=False)
print('Wrote:', baseline_csv)
print('Wrote:', baseline_table)
display(baseline_df)


## Component And Claim Evidence Ledger

This ledger connects the current artifacts to the claims that can and cannot be made after notebooks 02 and 03. It is designed for rebuttal traceability.


In [ ]:
component_rows = [
    {
        'claim_id': 'C01',
        'claim_topic': 'fair baseline superiority',
        'current_status': 'blocked_fair_baselines_missing',
        'supporting_artifacts': str(revision_root / 'metrics' / 'baseline_summary.csv'),
        'safe_wording': 'Do not claim improvement over fair baselines until Raw Mamba, ResNet1D/CNN, MiniRocket-only, and HRV-only rows are complete.',
    },
    {
        'claim_id': 'C02',
        'claim_topic': 'fixed-threshold ranking-decision gap',
        'current_status': 'supported_by_calibration_artifacts_with_limitations',
        'supporting_artifacts': '; '.join([
            str(calibration_path),
            str(revision_root / 'figures' / 'reliability_oof_final_ema_predictions.png'),
            str(revision_root / 'tables' / 'reliability_bins_oof_final_ema_predictions.csv'),
        ]),
        'safe_wording': 'Frozen OOF shows non-random ranking signal but low fixed-threshold recall/F1 and imperfect calibration; avoid clinical readiness wording.',
    },
    {
        'claim_id': 'C05',
        'claim_topic': 'Q=3 pooling operating point',
        'current_status': 'pending_pooling_sensitivity',
        'supporting_artifacts': str(revision_root / 'metrics' / 'pooling_sensitivity.csv'),
        'safe_wording': 'Run notebook 06 pooling sensitivity before justifying Q=3 beyond the frozen OOF contract.',
    },
    {
        'claim_id': 'C06',
        'claim_topic': 'protocol-faithful OOF evaluation',
        'current_status': 'partial_oof_supported',
        'supporting_artifacts': '; '.join([str(freeze_path), str(run_manifest_path), str(calibration_path)]),
        'safe_wording': 'OOF protocol is frozen and traceable; external outputs remain experimental until separate readiness gates pass.',
    },
]
component_df = pd.DataFrame(component_rows)
component_table = revision_root / 'tables' / 'table_component_checks.csv'
component_json = revision_root / 'metrics' / 'component_check_summary.json'
component_df.to_csv(component_table, index=False)
save_json(component_json, {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'claims': component_rows,
})
print('Wrote:', component_table)
print('Wrote:', component_json)
display(component_df)


## Heavy Baseline Guard

The official fair baseline runners are not implemented in this branch. This cell records the intended command surface and stops accidental placeholder execution.


In [ ]:
planned = {
    'MiniRocket-only': 'python scripts/revision/TBD_minirocket_only.py',
    'HRV-only': 'python scripts/revision/TBD_hrv_only.py',
    'ResNet1D/CNN': 'python scripts/revision/TBD_resnet1d_baseline.py',
    'Raw Mamba': 'python scripts/revision/TBD_raw_mamba_baseline.py',
}

for name, command in planned.items():
    script = Path(command.split()[1])
    print(f'{name:16s}: implemented={script.exists()} planned_command={command}')

if RUN_HEAVY_BASELINES:
    missing = [name for name, command in planned.items() if not Path(command.split()[1]).exists()]
    if missing:
        raise RuntimeError('Fair baseline runners are not implemented yet: ' + ', '.join(missing))
    for name, command in planned.items():
        run(command, log_path=f'reports/revision/logs/baseline_{name.lower().replace("/", "_").replace(" ", "_")}.log')
else:
    print('Heavy baseline execution disabled. Current notebook only writes traceable evidence and blocked baseline ledger rows.')


## Inventory And Stable Mirror

After notebook 04 writes evidence tables, freeze the revision artifact inventory and publish the artifacts to the Drive mirror for reuse.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/baseline_component_artifact_inventory.log',
)

mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/baseline_component_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'baseline_component_input_contract.json',
    revision_root / 'manifests' / 'baseline_component_plan_alignment.json',
    revision_root / 'metrics' / 'no_skill_reference_baselines.csv',
    revision_root / 'tables' / 'table_no_skill_reference_baselines.csv',
    revision_root / 'metrics' / 'baseline_summary.csv',
    revision_root / 'tables' / 'table_baselines.csv',
    revision_root / 'metrics' / 'component_check_summary.json',
    revision_root / 'tables' / 'table_component_checks.csv',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print('\nNotebook 04 complete. Current fair-baseline status: blocked until model-specific baseline runners are implemented and run under the same frozen OOF protocol.')
